<a href="https://colab.research.google.com/github/brysonje/DS_Training/blob/main/Wrangling/M1_handling_categorical_data_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
from sklearn import set_config

In [3]:
from sklearn.compose import make_column_selector as selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

In [6]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder

In [7]:
path = "https://www.openml.org/data/get_csv/1595261/phpMawTba.csv"
adult_census = pd.read_csv(path)
to_drop_list = ["fnlwgt", "education-num"]
adult_census = adult_census.drop(to_drop_list, axis = 1)

In [8]:
target_name = "class"
target = adult_census[target_name]
data = adult_census.drop(columns=[target_name])

In [9]:
adult_census.columns

Index(['age', 'workclass', 'education', 'marital-status', 'occupation',
       'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
       'hours-per-week', 'native-country', 'class'],
      dtype='object')

In [10]:
numerical_columns_selector = selector(dtype_exclude=object)
categorical_columns_selector = selector(dtype_include=object)

numerical_columns = numerical_columns_selector(data)
categorical_columns = categorical_columns_selector(data)

In [11]:
categorical_preprocessor = OneHotEncoder(handle_unknown="ignore")
numerical_preprocessor = StandardScaler()

In [12]:
preprocessor = ColumnTransformer([
    ('one-hot-encoder', categorical_preprocessor, categorical_columns),
    ('standard_scaler', numerical_preprocessor, numerical_columns)])

In [13]:
model = make_pipeline(preprocessor, LogisticRegression(max_iter=500))

In [14]:
set_config(display='diagram')
model

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('one-hot-encoder',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['workclass', 'education',
                                                   'marital-status',
                                                   'occupation', 'relationship',
                                                   'race', 'sex',
                                                   'native-country']),
                                                 ('standard_scaler',
                                                  StandardScaler(),
                                                  ['age', 'capital-gain',
                                                   'capital-loss',
                                                   'hours-per-week'])])),
                ('logisticregression', LogisticRegression(max_iter=500))])

In [15]:
data_train, data_test, target_train, target_test = train_test_split(
    data, target, random_state = 42)

In [16]:
_ = model.fit(data_train, target_train)

In [17]:
data_test.head()

,age,workclass,education,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
7762,56,Private,HS-grad,Divorced,Other-service,Unmarried,White,Female,0,0,40,United-States
23881,25,Private,HS-grad,Married-civ-spouse,Transport-moving,Own-child,Other,Male,0,0,40,United-States
30507,43,Private,Bachelors,Divorced,Prof-specialty,Not-in-family,White,Female,14344,0,40,United-States
28911,32,Private,HS-grad,Married-civ-spouse,Transport-moving,Husband,White,Male,0,0,40,United-States
19484,39,Private,Bachelors,Married-civ-spouse,Sales,Wife,White,Female,0,0,30,United-States


In [18]:
model.predict(data_test)[:5]

array([' <=50K', ' <=50K', ' >50K', ' <=50K', ' >50K'], dtype=object)

In [19]:
target_test[:5]

,class
7762,<=50K
23881,<=50K
30507,>50K
28911,<=50K
19484,<=50K


In [26]:
model.score(data_test, target_test)

0.8806813528785521

In [27]:
cv_results = cross_validate(model, data, target, cv=5)
cv_results

{'fit_time': array([1.58242297, 1.11222506, 1.1025176 , 1.09482479, 1.0876317 ]),
 'score_time': array([0.16584682, 0.17499256, 0.17706418, 0.18624353, 0.16658902]),
 'test_score': array([0.87081585, 0.87306787, 0.87192875, 0.87387387, 0.87981163])}

In [28]:
scores = cv_results["test_score"]
print("The mean cross-validation accuracy is: "
      f"{scores.mean():.3f} +/- {scores.std():.3f}")

The mean cross-validation accuracy is: 0.874 +/- 0.003


In [29]:
categorical_preprocessor = OrdinalEncoder(handle_unknown = "use_encoded_value",
                                          unknown_value = -1)

preprocessor = ColumnTransformer([('categorical', categorical_preprocessor,
                                   categorical_columns)],
    remainder = "passthrough")

model = make_pipeline(preprocessor, HistGradientBoostingClassifier())

In [30]:
%%time
_ = model.fit(data_train, target_train)

CPU times: user 1.04 s, sys: 16.2 ms, total: 1.06 s
Wall time: 1.06 s


In [31]:
model.score(data_test, target_test)

0.8797805257554664